In [33]:
from dotenv import load_dotenv
from anthropic import Anthropic
from DateTimeTool import get_current_datetime_schema, get_current_datetime

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system_prompt=None, temperature=None, toolSchemas=None):
    params = {
        "model": model,
        "max_tokens": 200,
        "messages": messages,
    }
    if system_prompt:
        params["system"] = system_prompt
    if temperature is not None:
        params["temperature"] = temperature
    if toolSchemas:
        params["tools"] = toolSchemas
    return client.messages.create(**params)

def UseTool(ToolName, Input):
    if ToolName == "get_current_datetime":
        return get_current_datetime(**Input)


def conversation(messages, continueConversation):

    while continueConversation:

        add_user_message(messages, input())

        response = chat(messages, toolSchemas=[get_current_datetime_schema])

        if response.content[0].type == "tool_use":
            ToolCallResult = UseTool(response.content[0].name, response.content[0].input)
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": response.content[0].id,
                    "content": ToolCallResult
                }]
            })
        elif response.content[0].type == "text":
            print(response.content[0].text)


In [ ]:
continueConversation = True
messages = []
result = conversation(messages, continueConversation)
result


The current time is **12:56:10** on **May 25, 2026**.
The current time is **12:56:10 PM** on **May 25, 2026**.


In [1]:
messages = []
add_user_message(messages, input())

response = chat(messages, toolSchemas=[get_current_datetime_schema])
response.content

NameError: name 'add_user_message' is not defined

In [32]:
response


#tool_block = response.content[0]
#tool_result = get_current_datetime(**tool_block.input)
#print("Tool returned:", tool_result)
#response.content[0]

Message(id='msg_01US2dedLEMbdMvNbt2yGVpC', container=None, content=[ToolUseBlock(id='toolu_016oCbyuvD4wndBHCFXNjgsr', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=601, output_tokens=69, server_tool_use=None, service_tier='standard'))

In [ ]:
#messages.append({"role": "assistant", "content": response.content})
#messages.append({
#    "role": "user",
#    "content": [{
#        "type": "tool_result",
#        "tool_use_id": tool_block.id,
#        "content": tool_result
#    }]
#})

#final = chat(messages, toolSchemas=[get_current_datetime_schema])
#print(final.content[0].text)